# k-NN — Classificação do Estado do Tabuleiro


## 1. Imports e Carregamento dos Splits


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

print("Carregando splits físicos...")
df_train = pd.read_csv('../../data/splits/train.csv')
df_val   = pd.read_csv('../../data/splits/val.csv')
df_test  = pd.read_csv('../../data/splits/test.csv')

X_train, y_train_raw = df_train.drop('classe', axis=1), df_train['classe']
X_val,   y_val_raw   = df_val.drop('classe', axis=1),   df_val['classe']
X_test,  y_test_raw  = df_test.drop('classe', axis=1),  df_test['classe']

le = LabelEncoder().fit(y_train_raw)
y_train = le.transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(y_test_raw)

print(f"Treino:    {len(X_train):4d} amostras")
print(f"Validação: {len(X_val):4d} amostras")
print(f"Teste:     {len(X_test):4d} amostras")


## 2. Busca do Melhor k (GridSearchCV no Treino)


In [ ]:
param_grid = {'n_neighbors': np.arange(1, 31)}
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

melhor_k = grid_search.best_params_['n_neighbors']
print(f"Melhor valor de k encontrado: {melhor_k}")

plt.figure(figsize=(10, 4))
scores = grid_search.cv_results_['mean_test_score']
plt.plot(param_grid['n_neighbors'], scores, marker='o', linestyle='dashed')
plt.title('Acurácia vs Valor de k (Cross-Validation)')
plt.xlabel('Valor de k'); plt.ylabel('Acurácia Média')
plt.grid(True); plt.show()


## 3. Treinamento do Modelo Final


In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=melhor_k)
knn_model.fit(X_train, y_train)
print(f"k-NN treinado com k={melhor_k}.")


## 4. Avaliação no Conjunto de Validação


In [ ]:
y_val_pred = knn_model.predict(X_val)

print("=== VALIDAÇÃO ===")
print(classification_report(
    le.inverse_transform(y_val),
    le.inverse_transform(y_val_pred),
    zero_division=0,
))

plt.figure(figsize=(7, 5))
cm_val = confusion_matrix(le.inverse_transform(y_val), le.inverse_transform(y_val_pred), labels=le.classes_)
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Matriz de Confusão — k-NN (Validação, k={melhor_k})')
plt.ylabel('Classe Real'); plt.xlabel('Classe Prevista')
plt.tight_layout(); plt.show()


## 5. Avaliação Final no Conjunto de Teste


In [ ]:
y_test_pred = knn_model.predict(X_test)

print("=== TESTE ===")
print(classification_report(
    le.inverse_transform(y_test),
    le.inverse_transform(y_test_pred),
    zero_division=0,
))

plt.figure(figsize=(7, 5))
cm_test = confusion_matrix(le.inverse_transform(y_test), le.inverse_transform(y_test_pred), labels=le.classes_)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Matriz de Confusão — k-NN (Teste, k={melhor_k})')
plt.ylabel('Classe Real'); plt.xlabel('Classe Prevista')
plt.tight_layout(); plt.show()


## 6. Persistência do Modelo


In [ ]:
caminho_modelo = '../../models/KNN'
os.makedirs(caminho_modelo, exist_ok=True)
joblib.dump(knn_model, f'{caminho_modelo}/knn_model.pkl')
joblib.dump(le,        f'{caminho_modelo}/knn_label_encoder.pkl')
print(f"Sucesso! Modelo k-NN salvo em: {caminho_modelo}")
